In [0]:
%load_ext autoreload
%autoreload 2
# Enables autoreload; learn more at https://docs.databricks.com/en/files/workspace-modules.html#autoreload-for-python-modules
# To disable autoreload; run %autoreload 0

# Leer desde la capa de bronce

In [0]:
from pyspark.sql.functions import col, when,trim,create_map, regexp_replace,lit,to_date,rlike, expr,coalesce, lower,broadcast, max, min, avg, stddev,row_number, sum as suma
from itertools import chain
from pyspark.sql.types import StringType
from pyspark.sql.window import Window
from clean import normalizar_fecha_spark,normalizar_nombre_columna


In [0]:
df_saps=spark.table("saps_ctes.bronze.datosctes_saps")
df_pato=spark.table("saps_ctes.silver.datosctes_consultas_patologia")
df_saps.limit(10).display()

# Realizar las transformaciones y limpieza

Revisemos primeros los valores para cada columna

In [0]:
df_saps.select("barrio").distinct().display()

Notamos que para los barrios tenemos valores nulos. Sigamos revisando las demás columnas.

In [0]:
df_saps.select("cargo").distinct().display()

Para el caso de los cargos también existen nulos

In [0]:
df_saps.select("contacto_telefono").distinct().display()

In [0]:
df_saps.select("responsable").distinct().display()

In [0]:
df_saps.select("saps").distinct().display()

In [0]:
df_saps.select("ubicacion").distinct().display()

Del análisis inicial podemos sacar las siguientes conclusiones:

- Hay nulos en los campos barrio y cargo.
- No tenemos un campo id como teníamos en las consultas.
- Debemos revisar que los nombres de los saps entre el listado de saps y consultas sea el mismo.

Lo que haremos es cargar los datos desde la capa de plata para las consultas y revisaremos qué los nombres de los saps sean correctos entre ambos. Además, veremos si existen saps en el listado de saps que no se encuentren en las consultas. Luego, crearemos una columna id_saps para el listado de saps usando los id_saps de la tabla de consultas. Finalmente, si existen saps que no estén entre los saps de las consultas, debemos darle un id para que no se queden sin ello.

# Revisando los saps de la lista de saps y las consultas.

NOTA: Hay un problema para mapear el nombre de Dr. ROMILIO MONZÓN a su versión Dr. Romilio Monzón. Por lo tanto, lo hago manualmente para acelerar el proceso de carga de datos. Esto se reparará posteriormente.

In [0]:

df_saps=df_saps.withColumn('saps',when(col('saps')=='Dr. ROMILIO MONZÓN','Dr. Romilio Monzón').otherwise(col('saps')))
df_saps_id=df_pato.select('id_saps','saps').distinct()  
df_saps=df_saps.join(broadcast(df_saps_id),on="saps",how="left").distinct().orderBy('saps')
df_saps.display()



In [0]:
df_pato.select('saps','id_saps').distinct().display()

Notamos que Dr. ROMILIO MONZÓN,Samun y Centro De Distribuciones De Vacunas no tienen id. Esto es porque son SAPS que no figuran en el dataframe de consultas. Debemos darles un id para no tener inconsistencias. Otro problema es que el nombre de Dr. ROMILIO MONZÓN no se mapea correctamente. Para acelerar el trabajo, el mapeo se hizo manualmente.

In [0]:
#primero, obtengo el máximo valor
max_id=df_saps.select(max('id_saps')).collect()[0][0]
df_saps_sin_id = (
    df_saps
    .filter((col("id_saps").isNull()) & (col("saps").isNotNull()))
    .select("saps")
    .distinct()
)


#creo la ventana para aplicar row_number
window = Window.orderBy("saps")


#aplico la función ventana
df_saps_nuevos_ids = (
    df_saps_sin_id
    .withColumn("nuevo_id_saps", row_number().over(window) + lit(max_id))
)




df_saps = (
    df_saps
    .join(df_saps_nuevos_ids, on="saps", how="left")
    .withColumn("id_saps", when(col("id_saps").isNotNull(), col("id_saps")).otherwise(col("nuevo_id_saps")))
    .drop("nuevo_id_saps")
)

df_saps.columns


In [0]:
df_saps.select(['id_saps','saps']).distinct().orderBy('saps').display()

Ya con los id completos, lo único que falta es completar los nulos con valores 'n/a' para las columnas barrio y cargo. 

In [0]:
df_saps=(
    df_saps.withColumn(
        'cargo',
        when(col('cargo').isNull(),'n/a')
        .otherwise(col('cargo'))
    )
    .withColumn(
        'barrio',
        when(col('barrio').isNull(),'n/a')
        .otherwise(col('barrio'))
    )
    .withColumn(
        'responsable',
        when(col('responsable').isNull(),'n/a')
        .otherwise(col('responsable'))
    )
    .withColumn(
        'ubicacion',
        when(col('ubicacion').isNull(),'n/a')
        .otherwise(col('ubicacion'))
    )
)

In [0]:
df_saps.printSchema()

# Pasar a la capa de plata

In [0]:
%sql
DROP TABLE IF EXISTS saps_ctes.silver.datosctes_saps;

In [0]:
df_saps.write.mode("overwrite").saveAsTable("saps_ctes.silver.datosctes_saps")

In [0]:
%sql
SELECT * FROM saps_ctes.silver.datosctes_saps limit 1